# Plant Disease Detection using Deep Learning

**Author:** Santanu Mandal  
**Project Type:** Internship / Machine Learning & Computer Vision  
**Domain:** Agriculture + Artificial Intelligence

This notebook builds an image-classification system that identifies selected tomato leaf conditions from the **PlantVillage** dataset. The project uses transfer learning with **MobileNetV2** and TensorFlow/Keras.

> **Important:** This is an educational computer-vision project. A prediction from the model should not be treated as a professional agricultural diagnosis.

## 1. Project Objectives

1. Load labeled plant-leaf images from the PlantVillage dataset.
2. Focus on five tomato classes: Bacterial Spot, Early Blight, Late Blight, Leaf Mold, and Healthy.
3. Preprocess and augment images.
4. Train a deep-learning image classifier using MobileNetV2 transfer learning.
5. Evaluate the model using accuracy, classification report and confusion matrix.
6. Save the trained model for later prediction.
7. Demonstrate prediction on a new leaf image.

In [ ]:
# Install dependencies if required
# Run this cell only if the packages are not already installed.
# !pip install -r requirements.txt

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import load_img, img_to_array

print("TensorFlow version:", tf.__version__)
print("TensorFlow Datasets version:", tfds.__version__)

## 2. Dataset

The project uses the PlantVillage dataset. The TensorFlow Datasets catalog describes PlantVillage as containing **54,303 labeled leaf images across 38 categories**. For a manageable internship demonstration, this notebook uses five tomato classes and limits the number of images per class.

Dataset reference:  
https://www.tensorflow.org/datasets/catalog/plant_village

Original dataset repository:  
https://github.com/spMohanty/PlantVillage-Dataset

In [ ]:
# Reproducibility and configuration
SEED = 42
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
MAX_IMAGES_PER_CLASS = 500
VALIDATION_SPLIT = 0.20
EPOCHS = 5

tf.random.set_seed(SEED)
np.random.seed(SEED)

CLASS_NAMES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___healthy"
]

# IDs from the published 38-class PlantVillage label list.
CLASS_IDS = [28, 29, 30, 31, 37]

print("Selected classes:")
for cid, name in zip(CLASS_IDS, CLASS_NAMES):
    print(cid, "->", name)

## 3. Load PlantVillage

The first run may download the dataset automatically. Because the complete dataset is large, the code filters to the five selected tomato classes and takes at most `MAX_IMAGES_PER_CLASS` images from each class.

In [ ]:
# Download/load PlantVillage through TensorFlow Datasets
(ds_all, ds_info), = tfds.load(
    "plant_village",
    split=["train"],
    as_supervised=True,
    with_info=True,
    shuffle_files=True
)

print("Total examples in PlantVillage:", ds_info.splits["train"].num_examples)

In [ ]:
# Build a smaller balanced dataset from the selected tomato classes.
class_datasets = []

for class_id in CLASS_IDS:
    class_ds = ds_all.filter(
        lambda image, label, class_id=class_id: tf.equal(label, class_id)
    ).take(MAX_IMAGES_PER_CLASS)
    class_datasets.append(class_ds)

selected_ds = class_datasets[0]
for extra_ds in class_datasets[1:]:
    selected_ds = selected_ds.concatenate(extra_ds)

selected_ds = selected_ds.shuffle(
    buffer_size=MAX_IMAGES_PER_CLASS * len(CLASS_IDS),
    seed=SEED,
    reshuffle_each_iteration=False
)

total_selected = tf.data.experimental.cardinality(selected_ds).numpy()
print("Selected images:", total_selected)

In [ ]:
# Preview a few images
plt.figure(figsize=(12, 8))

for i, (image, label) in enumerate(selected_ds.take(10)):
    plt.subplot(2, 5, i + 1)
    plt.imshow(image.numpy())
    plt.title(str(label.numpy()))
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Convert Labels and Split Data

The original PlantVillage labels are 0–37. We remap the five selected classes to local labels 0–4 so the final softmax layer has five outputs.

In [ ]:
# Map original class IDs -> local class IDs
table = tf.lookup.StaticHashTable(
    tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(CLASS_IDS, dtype=tf.int64),
        values=tf.constant(range(len(CLASS_IDS)), dtype=tf.int64)
    ),
    default_value=-1
)

def remap_label(image, label):
    return image, table.lookup(tf.cast(label, tf.int64))

selected_ds = selected_ds.map(remap_label, num_parallel_calls=tf.data.AUTOTUNE)

total_selected = tf.data.experimental.cardinality(selected_ds).numpy()
train_count = int(total_selected * (1 - VALIDATION_SPLIT))

train_raw = selected_ds.take(train_count)
val_raw = selected_ds.skip(train_count)

print("Training images:", train_count)
print("Validation images:", total_selected - train_count)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = train_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print(train_ds)
print(val_ds)

## 5. Data Augmentation

Small changes in rotation, zoom, translation and horizontal flipping help the model learn features that are less dependent on the exact camera position.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05)
], name="data_augmentation")

sample_images, sample_labels = next(iter(train_ds))

plt.figure(figsize=(12, 8))
for i in range(8):
    augmented = data_augmentation(tf.expand_dims(sample_images[i], 0), training=True)
    plt.subplot(2, 4, i + 1)
    plt.imshow(tf.clip_by_value(augmented[0], 0, 1))
    plt.title(CLASS_NAMES[int(sample_labels[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Build the Transfer-Learning Model

MobileNetV2 is used as a pretrained feature extractor. Its ImageNet weights provide general visual features, while the final classification layer is trained for the five tomato classes.

In [ ]:
try:
    base_model = MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet"
    )
    print("ImageNet pretrained weights loaded.")
except Exception as e:
    print("Could not download ImageNet weights. Using randomly initialized weights.")
    print("Reason:", e)
    base_model = MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights=None
    )

base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 7. Train the Model

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(10, 5))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

## 8. Evaluate the Model

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
y_true = []
y_pred = []

for batch_images, batch_labels in val_ds:
    probabilities = model.predict(batch_images, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    y_true.extend(batch_labels.numpy())
    y_pred.extend(predictions)

print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 9. Test Predictions on Sample Validation Images

In [ ]:
sample_images, sample_labels = next(iter(val_ds))
probabilities = model.predict(sample_images[:8], verbose=0)
predictions = np.argmax(probabilities, axis=1)

plt.figure(figsize=(14, 8))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(sample_images[i])
    true_name = CLASS_NAMES[int(sample_labels[i])]
    pred_name = CLASS_NAMES[int(predictions[i])]
    confidence = float(np.max(probabilities[i])) * 100
    plt.title(f"Pred: {pred_name}\nTrue: {true_name}\n{confidence:.1f}%")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 10. Save the Trained Model

In [ ]:
MODEL_PATH = "plant_disease_mobilenetv2.keras"
LABEL_PATH = "class_names.json"

model.save(MODEL_PATH)

with open(LABEL_PATH, "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, indent=2)

print("Saved:", MODEL_PATH)
print("Saved:", LABEL_PATH)

## 11. Predict a New Leaf Image

Set `IMAGE_PATH` to a JPG/PNG leaf image. The image should ideally contain one clear leaf similar to the training data.

Example:
```python
IMAGE_PATH = "my_leaf.jpg"
```

In [ ]:
IMAGE_PATH = "my_leaf.jpg"  # Change this to your image file

if os.path.exists(IMAGE_PATH):
    img = load_img(IMAGE_PATH, target_size=IMG_SIZE)
    arr = img_to_array(img) / 255.0
    probs = model.predict(np.expand_dims(arr, axis=0), verbose=0)[0]

    predicted_index = int(np.argmax(probs))
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = float(probs[predicted_index]) * 100

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{predicted_class}\nConfidence: {confidence:.2f}%")
    plt.show()

    print("Predicted class:", predicted_class)
    print(f"Confidence: {confidence:.2f}%")
else:
    print(f"Image not found: {IMAGE_PATH}")
    print("Place a JPG/PNG leaf image in the notebook folder and update IMAGE_PATH.")

## 12. Limitations and Future Scope

### Limitations
- PlantVillage images are largely collected under controlled conditions, so real-world performance can differ.
- This notebook intentionally uses only five tomato classes for a manageable internship demonstration.
- The model should not be treated as a definitive agricultural diagnosis.

### Future Scope
- Train on all 38 PlantVillage classes.
- Add field-condition datasets such as PlantDoc.
- Fine-tune the last MobileNetV2 layers.
- Build a Streamlit/Flask mobile-friendly web interface.
- Add disease-treatment information from verified agricultural sources.
- Deploy the model through IBM Cloud/watsonx or another cloud platform.
- Add image-quality checks and confidence thresholds before showing a prediction.

## 13. Conclusion

This project demonstrates an end-to-end AI workflow for plant disease classification: dataset loading, preprocessing, augmentation, transfer learning, training, evaluation, visualization, model saving, and image prediction. The same pipeline can be extended to more crops and disease classes.